# Evaluación fuera de muestra — RF h=5 new strategy (non-overlapping batches)

Evaluación financiera del modelo `rf_h5_expanding_discrete` en el periodo **1 mayo – 1 junio 2026**.

El modelo fue entrenado hasta finales de abril de 2026. Este notebook genera señales cada 5 días hábiles sobre el universo IBEX35 y evalúa el impacto económico en una cartera simulada de 100.000 €.

**Estrategias de referencia incluidas:**
- *Buy & Hold*: mantener posición en el índice IBEX35 durante todo el periodo
- *Clasificador aleatorio*: señales generadas aleatoriamente (1000 bootstrap, mismo calendario de lotes)
- *Momentum ingenuo*: predice que la dirección futura coincide con la tendencia de los últimos 5 días

**Supuestos operativos:**
- Señal al cierre del día *t* → entrada a apertura de *t+1* → salida al cierre de *t+5*
- El target del modelo es `log(close[t+5] / close[t])`, por lo que esta convención mide exactamente lo que el modelo aprendió a predecir
- **Lotes no solapados**: se toma señal cada 5 días hábiles; el capital está 100% en caja entre lotes
- Cartera inicial: 100.000 €
- Tamaño de posición uniforme entre todos los activos con señal activa en el lote
- Coste de transacción fijo: 10 pb por operación (round-trip = 20 pb)
- Con ~4 lotes en el periodo OOS, el ratio de Sharpe tiene elevada incertidumbre muestral

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from config import ARTIFACTS_PATH, load_env
from db.sqlite.queries_ohlcv import fetch_ohlcv
from db.utils_ohlcv import get_ibex_tickers, get_macro_tickers
from models.trees.features import ml_ready
from models.predict import _predict_tree

load_env()

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# Parámetros del experimento
OOS_START   = '2026-05-01'
OOS_END     = '2026-06-01'
INITIAL_NAV = 100_000.0
COST_BPS    = 10           # pb por lado → round-trip = 20 pb
COST_RT     = COST_BPS * 2 / 10_000
BATCH_STEP  = 5            # días hábiles entre lotes

## 1. Carga del artefacto y datos OHLCV

In [ ]:
artifact = joblib.load(ARTIFACTS_PATH / 'rf_h5_expanding_discrete.pkl')

print(f"Modelo         : {artifact['model_key']}")
print(f"Horizonte      : h={artifact['horizon']}")
print(f"Modo CV        : {artifact['mode']}")
print(f"ft_type        : {artifact['ft_type']}")
print(f"Entrenado hasta: {artifact['train_end']}")
print(f"Nº features    : {len(artifact['features'])}")
print(f"CV balanced acc: {artifact['cv_summary'].get('balanced_accuracy_mean', 'n/a'):.4f}")

In [ ]:
ibex_tickers  = get_ibex_tickers()
macro_tickers = get_macro_tickers()

df_micro_raw = fetch_ohlcv(ibex_tickers)
df_micro_raw = df_micro_raw[df_micro_raw['volume'] > 0].reset_index(drop=True)
df_macro_raw = fetch_ohlcv(macro_tickers)

print(f"OHLCV filas    : {len(df_micro_raw):,}")
print(f"Rango fechas   : {df_micro_raw['date'].min().date()} → {df_micro_raw['date'].max().date()}")
print(f"Tickers        : {df_micro_raw['ticker'].nunique()}")

## 2. Ingeniería de características y generación de señales OOS

In [ ]:
ft_type   = artifact['ft_type']
horizon   = artifact['horizon']   # 5
train_end = pd.Timestamp(artifact['train_end'])

df_macro_arg = df_macro_raw if ft_type == 'macro' else None
df_f, X_f, _, mask_f, _ = ml_ready(horizon, df_micro_raw, df_macro=df_macro_arg, ft_type=ft_type)

meta = df_f.loc[mask_f, ['ticker', 'date', 'open']].copy().reset_index(drop=True)
X    = X_f.reset_index(drop=True)

print(f"Filas con features completas: {len(X):,}  |  {X.shape[1]} features")

In [ ]:
oos_mask = (meta['date'] > train_end) & \
           (meta['date'] >= OOS_START) & \
           (meta['date'] <  OOS_END)

X_oos    = X.loc[oos_mask.values]
meta_oos = meta.loc[oos_mask.values].reset_index(drop=True)

print(f"Filas OOS: {len(X_oos):,}")
print(f"Fechas   : {meta_oos['date'].min().date()} → {meta_oos['date'].max().date()}")
print(f"Días únicos: {meta_oos['date'].nunique()}")

In [ ]:
preds, probas = _predict_tree(artifact, X_oos)

signals = meta_oos[['ticker', 'date']].copy()
signals['pred']  = preds
signals['proba'] = probas

print(f"Señales generadas : {len(signals):,}")
print(f"  → pred=1 (long): {(signals['pred']==1).sum():,}")
print(f"  → pred=0 (no position / caja): {(signals['pred']==0).sum():,}")
signals.head(10)

## 3. Motor de backtesting (lotes no solapados)

Para cada lote de señal cada 5 días hábiles:
- Longs: tickers con `pred=1` en la fecha de señal *t*
- Entrada en la apertura de *t+1*, salida al cierre de *t+5*
- Esto alinea directamente con el target del modelo: `log(close[t+5] / close[t])`
- Retorno neto por acción: retorno bruto − coste round-trip (20 pb)
- Retorno de cartera: media equiponderada entre todas las posiciones del lote
- Si *t+5* cae fuera del rango de datos disponibles, el lote se omite
- En días sin lote activo, retorno = 0 (caja)

In [ ]:
open_price = (
    df_micro_raw[['ticker', 'date', 'open']]
    .dropna()
    .set_index(['ticker', 'date'])['open']
    .to_dict()
)

close_price = (
    df_micro_raw[['ticker', 'date', 'close']]
    .dropna()
    .set_index(['ticker', 'date'])['close']
    .to_dict()
)

all_dates   = sorted(df_micro_raw['date'].unique())
date_to_idx = {d: i for i, d in enumerate(all_dates)}


def next_open(ticker, signal_date, steps):
    idx = date_to_idx.get(signal_date)
    if idx is None or idx + steps >= len(all_dates):
        return np.nan
    return open_price.get((ticker, all_dates[idx + steps]), np.nan)


def next_close(ticker, signal_date, steps):
    idx = date_to_idx.get(signal_date)
    if idx is None or idx + steps >= len(all_dates):
        return np.nan
    return close_price.get((ticker, all_dates[idx + steps]), np.nan)


def make_batch_dates(signal_dates, step=BATCH_STEP):
    """Return sorted list of non-overlapping batch signal dates spaced `step` trading days apart."""
    sorted_dates = sorted(signal_dates.unique())
    if not sorted_dates:
        return []
    batch_dates = [sorted_dates[0]]
    last_idx = date_to_idx[sorted_dates[0]]
    for d in sorted_dates[1:]:
        if date_to_idx[d] >= last_idx + step:
            batch_dates.append(d)
            last_idx = date_to_idx[d]
    return batch_dates


def run_backtest(sig_df, cost_rt=COST_RT, entry_steps=1, exit_steps=5):
    """Retorna DataFrame con columnas: date, port_ret, n_positions, gross_ret.

    Lotes no solapados cada `BATCH_STEP` días hábiles.
    Entrada: apertura de t+entry_steps. Salida: cierre de t+exit_steps.
    """
    batch_dates = make_batch_dates(sig_df['date'], step=BATCH_STEP)
    records = []

    for date in batch_dates:
        grp   = sig_df[sig_df['date'] == date]
        longs = grp[grp['pred'] == 1]['ticker'].tolist()
        if not longs:
            continue

        day_gross, day_net = [], []
        for tkr in longs:
            p_in  = next_open(tkr, date, entry_steps)   # apertura t+1
            p_out = next_close(tkr, date, exit_steps)   # cierre t+5
            if np.isnan(p_in) or np.isnan(p_out) or p_in == 0:
                continue
            gross = p_out / p_in - 1
            day_gross.append(gross)
            day_net.append(gross - cost_rt)

        if day_net:
            records.append({
                'date':        date,
                'port_ret':    float(np.mean(day_net)),
                'gross_ret':   float(np.mean(day_gross)),
                'n_positions': len(day_net),
            })

    return pd.DataFrame(records).set_index('date').sort_index()

## 4. Métricas financieras

In [ ]:
RISK_FREE_DAILY = 0.035 / 252

def financial_metrics(ret_series, gross_series=None, label='', initial_nav=INITIAL_NAV):
    """Rentabilidad acumulada neta y bruta, NAV neto y bruto, pérdida máxima y Sharpe."""
    r = ret_series.dropna()
    if len(r) == 0:
        return {}

    cum_factor = float((1 + r).prod())
    ret_acum   = cum_factor - 1
    nav_neto   = round(cum_factor * initial_nav, 2)

    if gross_series is not None:
        g = gross_series.dropna()
        cum_gross  = float((1 + g).prod())
        ret_bruta  = round(cum_gross - 1, 4)
        nav_bruto  = round(cum_gross * initial_nav, 2)
    else:
        ret_bruta = ret_acum
        nav_bruto = nav_neto

    equity = (1 + r).cumprod() * initial_nav
    dd     = (equity - equity.cummax()) / equity.cummax()
    max_dd = float(dd.min())

    excess = r - RISK_FREE_DAILY
    sharpe = float(excess.mean() / excess.std() * np.sqrt(252)) if excess.std() > 0 else 0.0

    return {
        'label':           label,
        'ret_bruta':       ret_bruta,
        'ret_acumulada':   round(ret_acum, 4),
        'nav_bruto':       nav_bruto,
        'nav_neto':        nav_neto,
        'perdida_maxima':  round(max_dd, 4),
        'sharpe':          round(sharpe, 3),
        'n_dias':          len(r),
    }

## 5. Modelo RF — señales OOS (lotes no solapados)

In [ ]:
bt_rf = run_backtest(signals)
m_rf  = financial_metrics(bt_rf['port_ret'], bt_rf['gross_ret'], label='RF h=5 new strategy')

batch_dates_used = make_batch_dates(signals['date'])
print(f"=== Random Forest h=5 expanding discrete (lotes no solapados) ===")
print(f"  Lotes ejecutados       : {len(bt_rf)}")
print(f"  Fechas de lote         : {[str(d.date()) for d in batch_dates_used]}")
print(f"  Rentabilidad bruta     : {m_rf['ret_bruta']:+.2%}")
print(f"  Rentabilidad acumulada : {m_rf['ret_acumulada']:+.2%}")
print(f"  NAV bruto              : {m_rf['nav_bruto']:,.2f} €")
print(f"  NAV neto               : {m_rf['nav_neto']:,.2f} €")
print(f"  Pérdida máxima         : {m_rf['perdida_maxima']:.2%}")
print(f"  Ratio de Sharpe        : {m_rf['sharpe']:.3f}  ⚠ n={len(bt_rf)} lotes — alta incertidumbre")
print(f"  Posiciones medias/lote : {bt_rf['n_positions'].mean():.1f}")

## 6. Estrategias de referencia

### 6.1 Buy & Hold IBEX35

In [ ]:
ibex_raw = (
    df_macro_raw[df_macro_raw['ticker'] == '^IBEX']
    .copy().sort_values('date')
)
ibex_raw['bh_ret'] = ibex_raw['close'].pct_change()

ibex_oos = ibex_raw[
    (ibex_raw['date'] >= OOS_START) & (ibex_raw['date'] < OOS_END)
].set_index('date').dropna(subset=['bh_ret'])

m_bh = financial_metrics(ibex_oos['bh_ret'], label='Buy & Hold IBEX35')

print("=== Buy & Hold IBEX35 ===")
print(f"  Rentabilidad bruta     : {m_bh['ret_bruta']:+.2%}")
print(f"  Rentabilidad acumulada : {m_bh['ret_acumulada']:+.2%}")
print(f"  NAV bruto              : {m_bh['nav_bruto']:,.2f} €")
print(f"  NAV neto               : {m_bh['nav_neto']:,.2f} €")
print(f"  Pérdida máxima         : {m_bh['perdida_maxima']:.2%}")
print(f"  Ratio de Sharpe        : {m_bh['sharpe']:.3f}")

### 6.2 Clasificador aleatorio (1000 bootstrap, mismo calendario de lotes)

In [ ]:
rng = np.random.default_rng(42)
rand_metrics = []

for _ in range(1000):
    r_sig = signals.copy()
    r_sig['pred']  = rng.integers(0, 2, size=len(r_sig))
    r_sig['proba'] = 0.5
    bt_r = run_backtest(r_sig)
    if not bt_r.empty:
        m = financial_metrics(bt_r['port_ret'], bt_r['gross_ret'])
        if m:
            rand_metrics.append(m)

rand_rets        = [m['ret_acumulada'] for m in rand_metrics]
rand_rets_brutos = [m['ret_bruta']     for m in rand_metrics]
rand_sharpes     = [m['sharpe']        for m in rand_metrics]
rand_dds         = [m['perdida_maxima'] for m in rand_metrics]

m_rand = {
    'label':          'Clasificador aleatorio (mediana)',
    'ret_bruta':      round(float(np.median(rand_rets_brutos)), 4),
    'ret_acumulada':  round(float(np.median(rand_rets)), 4),
    'nav_bruto':      round((1 + float(np.median(rand_rets_brutos))) * INITIAL_NAV, 2),
    'nav_neto':       round((1 + float(np.median(rand_rets))) * INITIAL_NAV, 2),
    'perdida_maxima': round(float(np.median(rand_dds)), 4),
    'sharpe':         round(float(np.median(rand_sharpes)), 3),
    'n_dias':         None,
}

print("=== Clasificador aleatorio (mediana bootstrap 1000, lotes no solapados) ===")
print(f"  Rentabilidad bruta     : {m_rand['ret_bruta']:+.2%}")
print(f"  Rentabilidad acumulada : {m_rand['ret_acumulada']:+.2%}")
print(f"  NAV bruto              : {m_rand['nav_bruto']:,.2f} €")
print(f"  NAV neto               : {m_rand['nav_neto']:,.2f} €")
print(f"  Pérdida máxima         : {m_rand['perdida_maxima']:.2%}")
print(f"  Ratio de Sharpe        : {m_rand['sharpe']:.3f}")
print(f"  p5 Sharpe / p95 Sharpe : {np.percentile(rand_sharpes,5):.3f} / {np.percentile(rand_sharpes,95):.3f}")

### 6.3 Momentum ingenuo (tendencia 5 días)

In [ ]:
mom_df = (
    df_micro_raw.sort_values(['ticker', 'date'])
    .assign(past_ret_5=lambda d: d.groupby('ticker')['close'].pct_change(5))
    .assign(pred=lambda d: (d['past_ret_5'] > 0).astype(int))
)

oos_dates = signals[['ticker', 'date']]
mom_sig = mom_df.merge(oos_dates, on=['ticker', 'date'])[['ticker', 'date', 'pred']].copy()
mom_sig['proba'] = mom_sig['pred'].astype(float)

bt_mom = run_backtest(mom_sig)
m_mom  = financial_metrics(bt_mom['port_ret'], bt_mom['gross_ret'], label='Momentum 5d')

print("=== Momentum ingenuo (5 días, lotes no solapados) ===")
print(f"  Rentabilidad bruta     : {m_mom['ret_bruta']:+.2%}")
print(f"  Rentabilidad acumulada : {m_mom['ret_acumulada']:+.2%}")
print(f"  NAV bruto              : {m_mom['nav_bruto']:,.2f} €")
print(f"  NAV neto               : {m_mom['nav_neto']:,.2f} €")
print(f"  Pérdida máxima         : {m_mom['perdida_maxima']:.2%}")
print(f"  Ratio de Sharpe        : {m_mom['sharpe']:.3f}")

## 7. Tabla resumen comparativa

In [ ]:
all_metrics = [m_rf, m_bh, m_rand, m_mom]

df_summary = pd.DataFrame(all_metrics).set_index('label')[[
    'ret_bruta', 'ret_acumulada', 'nav_bruto', 'nav_neto', 'perdida_maxima', 'sharpe'
]].rename(columns={
    'ret_bruta':       'Rent. bruta',
    'ret_acumulada':   'Rent. neta',
    'nav_bruto':       'NAV bruto (€)',
    'nav_neto':        'NAV neto (€)',
    'perdida_maxima':  'Pérdida máxima',
    'sharpe':          'Sharpe',
})

display(
    df_summary.style
    .format({
        'Rent. bruta':      '{:+.2%}',
        'Rent. neta':       '{:+.2%}',
        'NAV bruto (€)':    '{:,.2f}',
        'NAV neto (€)':     '{:,.2f}',
        'Pérdida máxima':   '{:.2%}',
        'Sharpe':           '{:.3f}',
    })
    .background_gradient(subset=['Sharpe'],      cmap='RdYlGn', vmin=-1.5, vmax=1.5)
    .background_gradient(subset=['Rent. neta'],  cmap='RdYlGn', vmin=-0.05, vmax=0.05)
)

## 8. Curvas de rentabilidad acumulada

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

# RF — step plot: NAV stays flat between batches
cum_rf = (1 + bt_rf['port_ret']).cumprod() * INITIAL_NAV
# Reindex to full OOS calendar so flat periods are visible
oos_cal = pd.date_range(OOS_START, pd.Timestamp(OOS_END) - pd.Timedelta(days=1), freq='B')
cum_rf_full = cum_rf.reindex(oos_cal, method='ffill').fillna(INITIAL_NAV)
ax.step(cum_rf_full.index, cum_rf_full.values, lw=2.0, color='steelblue',
        where='post', label='RF h=5 new strategy')

# Buy & Hold
if not ibex_oos.empty:
    cum_bh = (1 + ibex_oos['bh_ret']).cumprod() * INITIAL_NAV
    ax.plot(cum_bh.index, cum_bh.values, lw=1.8, ls=':', color='black', label='Buy & Hold IBEX35')

# Momentum
if not bt_mom.empty:
    cum_mom = (1 + bt_mom['port_ret']).cumprod() * INITIAL_NAV
    cum_mom_full = cum_mom.reindex(oos_cal, method='ffill').fillna(INITIAL_NAV)
    ax.step(cum_mom_full.index, cum_mom_full.values, lw=1.5, ls='--', color='darkorange',
            where='post', label='Momentum 5d')

ax.axhline(INITIAL_NAV, color='grey', lw=0.8, ls='--', alpha=0.6)
ax.set_title(
    f'Rentabilidad acumulada — periodo OOS {OOS_START} / {OOS_END}\n'
    f'Cartera inicial: {INITIAL_NAV:,.0f} €  |  Coste: {COST_BPS} pb/lado  |  Lotes cada {BATCH_STEP} días hábiles'
)
ax.set_ylabel('NAV (€)')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.show()

## 9. Pérdida máxima (drawdown)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

for label, bt_series, color, ls in [
    ('RF h=5 new strategy', bt_rf['port_ret'],   'steelblue',  '-'),
    ('Buy & Hold IBEX35',   ibex_oos['bh_ret'],  'black',      ':'),
    ('Momentum 5d',         bt_mom['port_ret'],  'darkorange', '--'),
]:
    if bt_series.empty:
        continue
    equity = (1 + bt_series).cumprod()
    dd     = (equity - equity.cummax()) / equity.cummax() * 100
    ax.plot(dd.index, dd.values, lw=1.5, color=color, ls=ls, label=label)

ax.axhline(0, color='grey', lw=0.7)
ax.set_title('Pérdida máxima (%)')
ax.set_ylabel('Drawdown (%)')
ax.legend()
plt.tight_layout()
plt.show()

## 10. Retorno por lote bruto vs. neto (RF)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Retorno neto por lote
axes[0].bar(
    range(len(bt_rf)),
    bt_rf['port_ret'] * 100,
    color=['green' if r >= 0 else 'red' for r in bt_rf['port_ret']],
    alpha=0.7,
)
axes[0].set_xticks(range(len(bt_rf)))
axes[0].set_xticklabels([str(d.date()) for d in bt_rf.index], rotation=45, ha='right')
axes[0].axhline(0, color='black', lw=0.8)
axes[0].set_title('Retorno neto por lote de cartera (%)')
axes[0].set_ylabel('%')

# Nº posiciones por lote
axes[1].bar(
    range(len(bt_rf)),
    bt_rf['n_positions'],
    color='steelblue',
    alpha=0.7,
)
axes[1].set_xticks(range(len(bt_rf)))
axes[1].set_xticklabels([str(d.date()) for d in bt_rf.index], rotation=45, ha='right')
axes[1].set_title('Número de posiciones por lote')
axes[1].set_ylabel('N')

plt.tight_layout()
plt.show()

## 11. Distribución del clasificador aleatorio vs. modelo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Sharpe
axes[0].hist(rand_sharpes, bins=40, color='lightgrey', edgecolor='white', alpha=0.9, label='Aleatorio (1000)')
axes[0].axvline(m_rf['sharpe'], color='steelblue', lw=2.0, label=f'RF: {m_rf["sharpe"]:.3f}')
axes[0].axvline(np.percentile(rand_sharpes, 95), color='red', lw=1.2, ls='--',
                label=f'p95 aleatorio: {np.percentile(rand_sharpes,95):.3f}')
axes[0].set_title('Distribución Sharpe — bootstrap aleatorio')
axes[0].set_xlabel('Sharpe')
axes[0].legend(fontsize=8)

# Rentabilidad acumulada
axes[1].hist(rand_rets, bins=40, color='lightgrey', edgecolor='white', alpha=0.9, label='Aleatorio (1000)')
axes[1].axvline(m_rf['ret_acumulada'], color='steelblue', lw=2.0,
                label=f'RF: {m_rf["ret_acumulada"]:+.2%}')
axes[1].axvline(np.percentile(rand_rets, 95), color='red', lw=1.2, ls='--',
                label=f'p95 aleatorio: {np.percentile(rand_rets,95):+.2%}')
axes[1].set_title('Distribución rentabilidad acumulada — bootstrap aleatorio')
axes[1].set_xlabel('Rentabilidad acumulada')
axes[1].xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

pct_beat_sharpe = float((np.array(rand_sharpes) < m_rf['sharpe']).mean())
pct_beat_ret    = float((np.array(rand_rets)    < m_rf['ret_acumulada']).mean())
print(f"El modelo supera al {pct_beat_sharpe:.1%} de los clasificadores aleatorios en Sharpe")
print(f"El modelo supera al {pct_beat_ret:.1%} de los clasificadores aleatorios en rentabilidad")

## 12. Rentabilidad bruta acumulada (antes de costes)

In [ ]:
cum_net   = (1 + bt_rf['port_ret']).cumprod() * INITIAL_NAV
cum_gross = (1 + bt_rf['gross_ret']).cumprod() * INITIAL_NAV

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(cum_gross.index, cum_gross.values, lw=1.8, color='cornflowerblue', ls='--',
        label='Rentabilidad bruta (antes de costes)')
ax.plot(cum_net.index,   cum_net.values,   lw=2.0, color='steelblue',
        label='Rentabilidad neta (después de costes)')
ax.axhline(INITIAL_NAV, color='grey', lw=0.8, ls='--', alpha=0.6)
ax.fill_between(cum_gross.index, cum_net.values, cum_gross.values,
                alpha=0.15, color='grey', label='Impacto costes')
ax.set_title(f'Rentabilidad bruta vs. neta — RF h=5 new strategy  |  {COST_BPS} pb/lado')
ax.set_ylabel('NAV (€)')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.show()

total_cost_eur = float(cum_gross.iloc[-1] - cum_net.iloc[-1])
print(f"Impacto total de costes: {total_cost_eur:,.2f} €  ({total_cost_eur/INITIAL_NAV:+.2%} sobre cartera inicial)")